In [3]:
# Mount Drive + Imports + Data Loading

# ================== INSTALL ==================
!pip install -q transformers datasets scikit-learn accelerate

# ================== MOUNT DRIVE ==================
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

# ================== IMPORTS ==================
import os, json, torch
import pandas as pd
import numpy as np
import pickle
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report

# ================== LOAD DATA ==================
path = "/content/drive/MyDrive/colab_datasets/dataset.json"

if not os.path.exists(path):
    raise FileNotFoundError("Dataset not found. Check path!")

with open(path, "r") as f:
    data = json.load(f)

texts, labels = [], []

for key in data:
    entry = data[key]
    text = " ".join(entry["post_tokens"])

    # Majority voting
    ann_labels = [ann["label"] for ann in entry["annotators"]]
    final_label = max(set(ann_labels), key=ann_labels.count)

    texts.append(text)
    labels.append(final_label)

df = pd.DataFrame({"text": texts, "label": labels})

print("Sample:\n", df.head())

# ================== ENCODE LABELS ==================
le = LabelEncoder()
df["label"] = le.fit_transform(df["label"])
num_labels = len(le.classes_)

# ================== SPLIT ==================
train_texts, temp_texts, train_labels, temp_labels = train_test_split(
    df["text"], df["label"], test_size=0.2, stratify=df["label"], random_state=42
)

val_texts, test_texts, val_labels, test_labels = train_test_split(
    temp_texts, temp_labels, test_size=0.5, stratify=temp_labels, random_state=42
)

print("Train:", len(train_texts), "Val:", len(val_texts), "Test:", len(test_texts))

# ================== METRICS FUNCTION ==================
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)

    return {
        "accuracy": accuracy_score(labels, preds),
        "precision": precision_score(labels, preds, average="macro"),
        "recall": recall_score(labels, preds, average="macro"),
        "f1": f1_score(labels, preds, average="macro")
    }

save_path = "/content/drive/MyDrive/saved_models"
os.makedirs(save_path, exist_ok=True)

with open(f"{save_path}/label_encoder.pkl", "wb") as f:
    pickle.dump(le, f)

Mounted at /content/drive
Sample:
                                                 text       label
0  i dont think im getting my baby them white 9 h...      normal
1  we cannot continue calling ourselves feminists...      normal
2                      nawt yall niggers ignoring me      normal
3  <user> i am bit confused coz chinese ppl can n...  hatespeech
4  this bitch in whataburger eating a burger with...  hatespeech
Train: 16118 Val: 2015 Test: 2015


In [4]:
# BERT

from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback
)

# ================== TOKENIZER ==================
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(texts):
    return tokenizer(list(texts), padding=True, truncation=True, max_length=128)

train_enc = tokenize(train_texts)
val_enc   = tokenize(val_texts)
test_enc  = tokenize(test_texts)

# ================== DATASET ==================
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = list(labels)

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_enc, train_labels)
val_dataset   = HateDataset(val_enc, val_labels)
test_dataset  = HateDataset(test_enc, test_labels)

# ================== MODEL ==================
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased", num_labels=num_labels
)

# ================== TRAIN ==================
training_args = TrainingArguments(
    output_dir="./bert_base",
    learning_rate=2e-5,
    num_train_epochs=5,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(2)]
)

trainer.train()

# ================== RESULTS ==================
print("Validation:", trainer.evaluate())

print("\nTrain Performance:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Performance:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))

# ================== SAVE MODEL ==================
bert_path = f"{save_path}/bert_base"

trainer.save_model(bert_path)
tokenizer.save_pretrained(bert_path)

print("BERT model saved at:", bert_path)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

...

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Validation: {'eval_loss': 0.7265967726707458, 'eval_accuracy': 0.6898263027295285, 'eval_precision': 0.680305198302614, 'eval_recall': 0.6776792497620266, 'eval_f1': 0.6788209637315775, 'eval_runtime': 16.1427, 'eval_samples_per_second': 124.825, 'eval_steps_per_second': 15.611, 'epoch': 4.0}

Train Performance:
              precision    recall  f1-score   support

           0       0.85      0.86      0.85      5133
           1       0.84      0.89      0.86      6601
           2       0.74      0.67      0.71      4384

    accuracy                           0.82     16118
   macro avg       0.81      0.81      0.81     16118
weighted avg       0.82      0.82      0.82     16118


Test Performance:


              precision    recall  f1-score   support

           0       0.73      0.70      0.72       642
           1       0.71      0.76      0.73       825
           2       0.52      0.48      0.50       548

    accuracy                           0.67      2015
   macro avg       0.65      0.65      0.65      2015
weighted avg       0.66      0.67      0.67      2015



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT model saved at: /content/drive/MyDrive/saved_models/bert_base


In [5]:
# Bert and Class Weights

from sklearn.utils.class_weight import compute_class_weight
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
from torch import nn

model_ckpt = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def tokenize_fn(texts):
    return tokenizer(list(texts), truncation=True, max_length=128)

# Dataset
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_fn(texts)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_texts, train_labels)
val_dataset   = HateDataset(val_texts, val_labels)
test_dataset  = HateDataset(test_texts, test_labels)

# Class weights
weights = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = torch.tensor(weights, dtype=torch.float).to("cuda")

# Custom Trainer
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        loss_fn = nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits.view(-1, num_labels), labels.view(-1))

        return (loss, outputs) if return_outputs else loss

model = AutoModelForSequenceClassification.from_pretrained(model_ckpt, num_labels=num_labels)

training_args = TrainingArguments(
    output_dir="./bert_weighted",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# ================== RESULTS ==================
print("\nValidation Metrics:")
print(trainer.evaluate())

print("\nTrain Report:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Report:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))


# ================== SAVE MODEL ==================
bert_weighted_path = f"{save_path}/bert_weighted"

trainer.save_model(bert_weighted_path)
tokenizer.save_pretrained(bert_weighted_path)

print("Weighted BERT saved at:", bert_weighted_path)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1
1,0.869238,0.760803,0.678908,0.674882,0.673806,0.672838
2,0.697009,0.742419,0.675434,0.678931,0.676329,0.672750
3,0.554856,0.818105,0.673945,0.661747,0.666297,0.662944
4,0.408811,0.939638,0.660050,0.652826,0.654426,0.652050
5,0.273258,1.120922,0.661538,0.650223,0.651691,0.650769
6,0.195834,1.304105,0.659553,0.647329,0.651559,0.647889
7,0.129727,1.485160,0.655087,0.651842,0.652233,0.648823
8,0.099300,1.652590,0.664020,0.651980,0.655194,0.652467
9,0.072575,1.823653,0.662035,0.656545,0.656718,0.654920
10,0.060645,1.881082,0.663524,0.656025,0.657078,0.655221


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...


Validation Metrics:


{'eval_loss': 0.7434478998184204, 'eval_accuracy': 0.678908188585608, 'eval_precision': 0.6808732095543522, 'eval_recall': 0.6791818723879194, 'eval_f1': 0.6757378944075478, 'eval_runtime': 4.8644, 'eval_samples_per_second': 414.231, 'eval_steps_per_second': 51.805, 'epoch': 10.0}

Train Report:
              precision    recall  f1-score   support

           0       0.84      0.85      0.85      5133
           1       0.90      0.79      0.84      6601
           2       0.67      0.78      0.72      4384

    accuracy                           0.81     16118
   macro avg       0.80      0.81      0.80     16118
weighted avg       0.82      0.81      0.81     16118


Test Report:


              precision    recall  f1-score   support

           0       0.73      0.71      0.72       642
           1       0.74      0.63      0.68       825
           2       0.46      0.58      0.52       548

    accuracy                           0.64      2015
   macro avg       0.65      0.64      0.64      2015
weighted avg       0.66      0.64      0.65      2015



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Weighted BERT saved at: /content/drive/MyDrive/saved_models/bert_weighted


In [6]:
# XLM-RoBERTa

# ================== XLM-R ==================
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    Trainer,
    TrainingArguments,
    DataCollatorWithPadding
)
model_ckpt = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

# ================== DATASET ==================
class HateDataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenize_fn(texts)
        self.labels = labels.tolist()

    def __getitem__(self, idx):
        item = {k: torch.tensor(v[idx]) for k,v in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        return len(self.labels)

train_dataset = HateDataset(train_texts, train_labels)
val_dataset   = HateDataset(val_texts, val_labels)
test_dataset  = HateDataset(test_texts, test_labels)

# ================== MODEL ==================

model = AutoModelForSequenceClassification.from_pretrained(
    model_ckpt, num_labels=num_labels
)

training_args = TrainingArguments(
    output_dir="./xlm_results",
    learning_rate=2e-5,
    num_train_epochs=10,
    per_device_train_batch_size=16,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    fp16=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics
)

trainer.train()

# ================== RESULTS ==================
print("\nValidation Metrics:")
print(trainer.evaluate())

print("\nTrain Report:")
train_preds = trainer.predict(train_dataset)
print(classification_report(train_labels, np.argmax(train_preds.predictions, axis=1)))

print("\nTest Report:")
test_preds = trainer.predict(test_dataset)
print(classification_report(test_labels, np.argmax(test_preds.predictions, axis=1)))

# ================== SAVE MODEL ==================
xlm_path = f"{save_path}/xlm_roberta"

trainer.save_model(xlm_path)
tokenizer.save_pretrained(xlm_path)

print("XLM-R model saved at:", xlm_path)

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.12G [00:00<?, ?B/s]

...


Validation Metrics:


{'eval_loss': 0.7650797367095947, 'eval_accuracy': 0.658560794044665, 'eval_precision': 0.6516155745273471, 'eval_recall': 0.6296689526865994, 'eval_f1': 0.6251329669108978, 'eval_runtime': 4.5848, 'eval_samples_per_second': 439.499, 'eval_steps_per_second': 54.965, 'epoch': 10.0}

Train Report:
              precision    recall  f1-score   support

           0       0.77      0.80      0.79      5133
           1       0.68      0.87      0.76      6601
           2       0.68      0.36      0.47      4384

    accuracy                           0.71     16118
   macro avg       0.71      0.68      0.67     16118
weighted avg       0.71      0.71      0.69     16118


Test Report:


              precision    recall  f1-score   support

           0       0.71      0.71      0.71       642
           1       0.64      0.82      0.72       825
           2       0.60      0.35      0.44       548

    accuracy                           0.66      2015
   macro avg       0.65      0.63      0.62      2015
weighted avg       0.65      0.66      0.64      2015



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

XLM-R model saved at: /content/drive/MyDrive/saved_models/xlm_roberta


In [7]:
# T5 Model

from transformers import T5Tokenizer, T5ForConditionalGeneration, Trainer, TrainingArguments
from sklearn.metrics import classification_report, accuracy_score

tokenizer = T5Tokenizer.from_pretrained("t5-small")

class T5Dataset(torch.utils.data.Dataset):
    def __init__(self, texts, labels):
        self.inputs = tokenizer(
            ["classify: " + str(t) for t in texts],
            padding="max_length",
            truncation=True,
            max_length=256
        )
        self.targets = tokenizer(
            list(map(str, labels)),
            padding="max_length",
            truncation=True,
            max_length=10
        )

    def __getitem__(self, idx):
        labels = torch.tensor(self.targets["input_ids"][idx])
        labels[labels == tokenizer.pad_token_id] = -100

        return {
            "input_ids": torch.tensor(self.inputs["input_ids"][idx]),
            "attention_mask": torch.tensor(self.inputs["attention_mask"][idx]),
            "labels": labels
        }

    def __len__(self):
        return len(self.inputs["input_ids"])

train_dataset = T5Dataset(train_texts.tolist(), train_labels.tolist())
val_dataset   = T5Dataset(val_texts.tolist(), val_labels.tolist())
test_dataset  = T5Dataset(test_texts.tolist(), test_labels.tolist())

model = T5ForConditionalGeneration.from_pretrained("t5-small")

training_args = TrainingArguments(
    output_dir="./t5_results",
    learning_rate=3e-5,
    num_train_epochs=10,
    per_device_train_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    report_to="none"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset
)

trainer.train()

# ================== PREDICTION FUNCTION ==================
def get_preds(dataset):
    model.eval()
    preds = []

    for batch in dataset:
        input_ids = batch["input_ids"].unsqueeze(0).to(model.device)
        attention_mask = batch["attention_mask"].unsqueeze(0).to(model.device)

        outputs = model.generate(input_ids=input_ids, attention_mask=attention_mask, max_length=5)
        pred = tokenizer.decode(outputs[0], skip_special_tokens=True)

        preds.append(int(pred) if pred.isdigit() else -1)

    return preds

# ================== RESULTS ==================
test_preds = get_preds(test_dataset)
test_true = test_labels.tolist()

print("\nTest Classification Report:")
print(classification_report(test_true, test_preds))

# ================== TRAIN RESULTS ==================
train_preds = get_preds(train_dataset)
train_true = train_labels.tolist()

print("\nTrain Classification Report:")
print(classification_report(train_true, train_preds))
print("Train Accuracy:", accuracy_score(train_true, train_preds))


# ================== VALIDATION RESULTS ==================
val_preds = get_preds(val_dataset)
val_true = val_labels.tolist()

print("\nValidation Classification Report:")
print(classification_report(val_true, val_preds))
print("Validation Accuracy:", accuracy_score(val_true, val_preds))

# ================== SAVE MODEL ==================
t5_path = f"{save_path}/t5_model"

trainer.save_model(t5_path)
tokenizer.save_pretrained(t5_path)

print("T5 model saved at:", t5_path)

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight', 'lm_head.weight'].



Test Classification Report:
              precision    recall  f1-score   support

           0       0.74      0.64      0.69       642
           1       0.64      0.80      0.71       825
           2       0.54      0.41      0.46       548

    accuracy                           0.64      2015
   macro avg       0.64      0.62      0.62      2015
weighted avg       0.64      0.64      0.64      2015


Train Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.73      0.76      5133
           1       0.67      0.83      0.74      6601
           2       0.60      0.44      0.50      4384

    accuracy                           0.69     16118
   macro avg       0.69      0.67      0.67     16118
weighted avg       0.69      0.69      0.68     16118

Train Accuracy: 0.691338875791041

Validation Classification Report:
              precision    recall  f1-score   support

           0       0.78      0.70      0.74       641
 

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

T5 model saved at: /content/drive/MyDrive/saved_models/t5_model
